### **Exercise: Sentiment Analysis and Key Insights Extraction from Ford Car Reviews**

### **Problem Statement:**
You have been provided with a dataset containing Ford car reviews. Your task is to use LangChain and the concepts you’ve learned to perform the following tasks:

1. **Sentiment Analysis**: Analyze the sentiment of each review, categorize it as positive, neutral, or negative, and store the result.
2. **Key Insights Extraction**: Extract key pieces of information from each review, such as the pros and cons mentioned, and the specific features the reviewer liked or disliked (e.g., vehicle performance, comfort, price).

You will build a LangChain-based solution that leverages language models to automatically extract this information and provide a structured summary of the reviews. 

---
### **Steps to Solve:**

#### **Step 1: Load the Dataset**
- The dataset file is named `ford_car_reviews.csv` and is sourced from Kaggle: [Edmunds Consumer Car Ratings and Reviews](https://www.kaggle.com/datasets/ankkur13/edmundsconsumer-car-ratings-and-reviews).
- For this exercise, **limit the data to the first 25 records**. This can be achieved by using `df.head(25)` or `df.iloc[:25]` when loading the data into a DataFrame.

#### **Step 2: Define the Sentiment Analysis Task**
- Use LangChain to create a pipeline to classify the sentiment of each review.
- Define prompts that can guide the model to evaluate the sentiment. For example:
  - "Given the following car review, classify the sentiment as positive, neutral, or negative."

#### **Step 3: Key Insights Extraction**
- Use LangChain to create a pipeline to extract pros, cons, and notable features from each review. Define prompts such as:
  - "What are the pros and cons of the vehicle described in the following review?"
  - "What specific features of the vehicle does the reviewer like or dislike?"

#### **Step 4: Update the DataFrame with New Information**
- Run the pipeline for each review and collect the sentiment and insights.
- Once the analysis and extraction are complete, update the original DataFrame with additional columns to include:
  - Sentiment (positive, neutral, negative)
  - Pros
  - Cons
  - Liked_Features
  - Disliked_Features

---

### **Example Output:**

```json
{
  "Review_Date": "03/07/13",
  "Vehicle_Title": "2006 Ford Mustang Coupe",
  "Review_Text": "With the expected arrival of our 6th child...",
  "Rating": 4.125,
  "Sentiment": "Positive",
  "Pros": "Good driving experience, Large seating capacity, Great options",
  "Cons": "None mentioned",
  "Liked_Features": ["Driving experience", "Seating capacity", "Options available"],
  "Disliked_Features": []
}
```

In [9]:
import os
import getpass
import time
import pandas as pd
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(
    model_name,
    model_provider="groq",
    temperature=0,
    max_tokens=1024,
    reasoning_effort="low",
)

from pathlib import Path

DATA_PATH = Path("ford_car_reviews.csv")
df = pd.read_csv(DATA_PATH, engine="python").head(25).copy()
df["Review"] = df["Review"].fillna("").astype(str)

print(f"Loaded {len(df)} reviews from {DATA_PATH}")
df[["Vehicle_Title", "Review_Title", "Rating", "Review"]].head(3)

class ReviewInsights(BaseModel):
    sentiment: str = Field(description="Overall sentiment: positive, neutral, or negative")
    pros: str = Field(description="Main advantages or positive points mentioned; use 'None mentioned' when absent")
    cons: str = Field(description="Main disadvantages or negative points mentioned; use 'None mentioned' when absent")
    liked_features: list[str] = Field(description="Specific vehicle features the reviewer likes; use an empty list when absent")
    disliked_features: list[str] = Field(description="Specific vehicle features the reviewer dislikes; use an empty list when absent")


parser = JsonOutputParser(pydantic_object=ReviewInsights)

review_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You analyze Ford car reviews. Return only valid JSON matching the requested schema.
Classify sentiment using exactly one lowercase label: positive, neutral, or negative.
Use only information stated or clearly implied by the review. Do not invent details.
{format_instructions}""",
    ),
    ("human", "Review:\n{review}"),
]).partial(format_instructions=parser.get_format_instructions())

review_chain = review_prompt | llm | parser

def analyze_review(review: str, max_retries: int = 4) -> dict:
    """Run the LangChain extraction chain for one review with rate-limit backoff."""
    for attempt in range(max_retries):
        try:
            result = review_chain.invoke({"review": review})
            result["sentiment"] = result["sentiment"].strip().lower()
            if result["sentiment"] not in {"positive", "neutral", "negative"}:
                result["sentiment"] = "neutral"
            return result
        except Exception as error:
            if "rate_limit" not in str(error).lower() or attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)

# A single-row smoke test confirms the model and parser contract before the full run.
first_result = analyze_review(df.iloc[0]["Review"])
first_result

analysis_results = [first_result] + [analyze_review(review) for review in df["Review"].iloc[1:]]

analysis_df = pd.DataFrame(analysis_results)
df["Sentiment"] = analysis_df["sentiment"]
df["Pros"] = analysis_df["pros"]
df["Cons"] = analysis_df["cons"]
df["Liked_Features"] = analysis_df["liked_features"]
df["Disliked_Features"] = analysis_df["disliked_features"]

required_columns = {"Sentiment", "Pros", "Cons", "Liked_Features", "Disliked_Features"}
assert len(df) == 25
assert required_columns.issubset(df.columns)
assert set(df["Sentiment"].unique()).issubset({"positive", "neutral", "negative"})

df[["Review_Date", "Vehicle_Title", "Review", "Rating", *sorted(required_columns)]].head(10)


Loaded 25 reviews from ford_car_reviews.csv


,Review_Date,Vehicle_Title,Review,Rating,Cons,Disliked_Features,Liked_Features,Pros,Sentiment
0,on 06/06/18 14:19 PM (PDT),2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,Doesn’t disappoint,5.000,None mentioned,[],[],None mentioned,positive
1,on 08/12/17 06:06 AM (PDT),2006 Ford Mustang Coupe V6 Standard 2dr Coupe ...,I bought mine 4/17 with 98K. Have been wantin...,3.000,"Transmission is difficult to use, harsh ride, ...","[transmission, ride comfort, road noise]","[engine performance, appearance, driving enjoy...","Engine is fine, good power, great mileage, att...",neutral
2,on 06/15/17 05:43 AM (PDT),2006 Ford Mustang Coupe V6 Premium 2dr Coupe (...,There will always be a 05-09 mustang for sale...,5.000,None mentioned,[],[],"Affordable used Mustang, considered a great in...",positive
3,on 05/18/17 17:33 PM (PDT),2006 Ford Mustang Coupe V6 Deluxe 2dr Coupe (4...,I bought my car from an auction I work at ( A...,5.000,None mentioned,[],"[V6 engine, cold air injector, throttle body s...","The reviewer loves the car, praising its power...",positive
4,on 01/03/16 18:03 PM (PST),2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,I bought this car spankin new and i still am ...,5.000,Had to replace the alternator; normal wear ite...,[alternator],"[road handling, responsive performance]","Responsive handling that hugs the road, reliab...",positive
5,on 10/24/15 12:40 PM (PDT),2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,"Lots of problems with Ford these days, sensor...",3.000,"Lots of problems with sensors, cam phasers, an...","[sensors, cam phasers, solenoid]",[],None mentioned,negative
6,on 10/29/11 04:57 AM (PDT),2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,Bought mine used 20k on it and have added a S...,4.625,None mentioned,[],"[SCT tuner, Ford CAI, Flow master mufflers]","Great performance, decent fuel economy, good c...",positive
7,on 07/25/11 12:15 PM (PDT),2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,I bought my preowned 06 a few weeks back and ...,4.375,None mentioned,[],"[Airaid air filter system, Flowmaster exhaust]","Upgraded air filter system and exhaust, high h...",positive
8,on 07/21/11 11:28 AM (PDT),2006 Ford Mustang Coupe GT Deluxe 2dr Coupe (4...,I drive 50 miles each way to work and traded ...,3.500,"Engine knocking, loss of power, costly cam pha...","[cam phaser reliability, high repair costs]",[],None mentioned,negative
9,on 12/06/10 00:00 AM (PST),2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,This car is just awesome. The 4.6L V8 makes ...,4.625,None mentioned,[],"[4.6L V8 engine, stock exhaust sound, performa...","Powerful 4.6L V8 engine, pleasant rumbling exh...",positive
